# Test Stage 8 — Active feedback against a resonator whose answer is known

**What this stage checks:** the feedback system — the phase shifter driving a resonant
load — using the digital resonator as a stand-in for the trap's detection coil.

The real coil resonates near 40 MHz with a quality factor (Q) of about 180 at room
temperature. Testing feedback against it means a cryostat, cables, and a resonator whose
exact centre frequency and Q are themselves unknowns. The digital resonator sidesteps all
of that: it IS a resonator, built in logic, with a centre frequency and Q that this
notebook sets *exactly*. You choose the answer; the measurement must return it. Anything
else is the instrument — the same known-answer discipline that found the only real bug in
the acquisition system.

The loop closes entirely inside the chip, with no cables:

```
generator ─┐
           ├──▶ DAC value ──▶ [resonator] ──┬──▶ acquisition system (measures)
feedback ──┘                                │
    ▲                                       │
    └────────── phase shifter ◀─────────────┘
```

This reproduces the phase/gain sweeps of Herkenhoff *et al.*, Rev. Sci. Instrum. **92**,
103201 (2021): sweeping the feedback phase modulates the resonator's effective linewidth
(and shifts its centre), and sweeping the gain scales that modulation — the mechanism the
real experiment uses to narrow the detection resonance.

## What you need

| | |
|---|---|
| Stages 1–7 | Passing |
| **A bitstream containing the resonator** | Step 0 checks for it and refuses to continue on an older build |
| Signal generator / cables | **None.** Everything is internal |
| After any reboot | `sudo python3 diagnose_hp_ports.py --open-tz` is *not* needed here (no memory captures) — but run `check_system.py` as always |

## The predictions being tested

**Open loop.** The measured resonance must be a Lorentzian centred on the frequency asked
for, with the Q asked for (to within the coefficient rounding, which is computed exactly
in Step 0 — better than 1%).

**Phase sweep at fixed gain.** With loop gain K, feedback at phase phi relative to the
resonator's own oscillation changes the *damping*:

```
linewidth(phi)  =  linewidth0 x (1 - K cos(phi - phi_n))
centre(phi)     =  f0 + (linewidth0 x K / 2) x sin(phi - phi_n)
```

Two sinusoids, a quarter turn apart, and the centre modulation must be exactly **half**
the linewidth modulation — a ratio the physics fixes and the fit must reproduce. phi_n
absorbs the loop's transport delay (the emulated 119.87 ns is 286 degrees at 40 MHz).

**Gain sweep at the narrowing phase.** `linewidth/linewidth0 = 1 - K`, and K is
proportional to gain — a straight line whose zero crossing is the self-oscillation
threshold. Approaching it is how the real experiment gets its narrowest lines, and
knowing where it is keeps the sweep safely below it.

---

### Simulation or the real trap?

**This stage runs against the emulated coil** — a resonator built in logic whose centre
frequency and Q you set exactly. That is the entire point: every measurement has a known
answer, so a disagreement is the instrument's fault rather than the physics'.

On the real coil there is no known answer, and three things change:

| | Emulated coil | Real coil |
|---|---|---|
| Where the resonance sits | `reso.config(F0, Q_SET)` puts it there, and `reso.f0_q` reports it back | Nothing puts it anywhere. **Measure it first** with a wide coarse sweep, and use that number everywhere `reso.f0_q` appears |
| How wide it is | `LW = reso.f0_q / reso.q_q`, known in advance | Comes out of that same first sweep. It is a result, not an input |
| The PASS gates | Compare against the Q you asked for | Cannot — there is nothing to compare against. They become *reports* |

What survives on the real coil are the checks that do not need a known answer: does the
linewidth change in proportion to feedback gain, do the narrowing and broadening phases agree
with each other, and does a setting reproduce when re-applied cold. Those are the ones worth
keeping.

Finally, **do not call `reso.enable()`**. The resonator is off at power-on, and while it is
off the board passes the real input straight through. No rebuild and no new bitstream are
needed to switch between the two.


## Setup

> **Real coil — the one cell that changes most.** Step 0 below configures the emulated
> resonator and switches it into the signal path. For real measurements, delete the
> `reso.config(...)` and `reso.enable()` calls: leave the resonator disabled and the board
> listens to the actual input. Everything downstream that reads `reso.f0_q` then needs a
> measured number instead — see the table at the top.


In [ ]:
import os, mmap, struct, time
import numpy as np

FS_CLK = 125e6      # the one clock everything on the chip runs from [Hz]

F0     = 40.0e6     # centre frequency to emulate [Hz]  - the real coil is ~40 MHz
Q_SET  = 180.0      # quality factor to emulate         - the real coil is ~180 at rtp

SETTLE = 0.02       # wait after any retune [s]. The correction filter's group delay
                    # is ~2 ms at R = 1024 and the resonator rings up in microseconds,
                    # so 20 ms is comfortable for everything at once.
N_AVG  = 32         # register-read pairs averaged per measurement point

# =============================================================================
#  One small bus driver shared by all four blocks.
#  CRITICAL RULE (same as every notebook in this project): one 32-bit access
#  at a time. A bulk mmap slice becomes a burst the register bridge cannot
#  answer; it locks the bus and reboots the board.
# =============================================================================
class Bus(object):
    def __init__(self, base, span=0x1000):
        self.fd = os.open('/dev/mem', os.O_RDWR | os.O_SYNC)
        self.m  = mmap.mmap(self.fd, span, offset=base)
    def w(self, off, val):
        self.m[off:off + 4] = struct.pack('<I', val & 0xFFFFFFFF)
    def r(self, off):
        return struct.unpack('<I', self.m[off:off + 4])[0]


def s32(v):
    """Reinterpret a 32-bit register value as signed."""
    return v - (1 << 32) if v & 0x80000000 else v


# =============================================================================
#  ACQUISITION SYSTEM (0x40600000) - used here as a narrowband voltmeter.
#
#  The trick that makes the whole sweep simple: tune the acquisition's local
#  oscillator to EXACTLY the drive frequency. The signal then lands at 0 Hz,
#  where I and Q are steady numbers, and the PEEK registers can be read at
#  leisure. (At any other offset I and Q rotate, and two register reads land
#  on different samples - the exact artefact that once mimicked droop. At DC
#  that artefact cannot exist, which is why this method is trustworthy.)
#
#  Retuning is ONE register write: the "disable -> settings -> reset" ritual
#  is only needed when DECIMATION changes, because the decimator carries a
#  running total. Changing the mixer frequency mid-run is safe.
# =============================================================================
class Acq(Bus):
    def __init__(self):
        Bus.__init__(self, 0x40600000)
    def start(self, dec_log2=10):
        """Enable once: IN1 source, no test pattern, not armed, filter on."""
        self.w(0x00, 0)                    # disable while settings change
        self.w(0x08, dec_log2)             # decimation R = 2^dec_log2
        self.w(0x0C, 0)                    # source: channel A
        self.w(0x00, 1 | 2)                # enable + soft reset (one-shot)
        self.w(0x00, 1)                    # running
    def tune(self, f):
        """Point the local oscillator at f - the one-write retune."""
        self.w(0x04, int(round(f / FS_CLK * (1 << 32))) & 0xFFFFFFFF)
    def measure(self, navg=N_AVG):
        """Average navg I/Q pairs at DC; returns one complex number."""
        zi = 0.0; zq = 0.0
        for _ in range(navg):
            zi += s32(self.r(0x18))
            zq += s32(self.r(0x1C))
        return complex(zi / navg, zq / navg)


# =============================================================================
#  DIGITAL RESONATOR (0x40400000) - the plant. See red_pitaya_resonator.v.
#
#  Software sets two 18-bit coefficients, c = r cos(w0) and s = r sin(w0),
#  where w0 is the centre frequency as a fraction of the clock and r is the
#  pole radius that sets Q. Because the coefficients are ROUNDED to 18 bits,
#  the resonator the chip actually builds differs minutely from the one asked
#  for - so this driver computes the frequency and Q implied by the ROUNDED
#  values, and every later PASS/FAIL compares against those, not the ideals.
#  Comparing against the ideal would smear coefficient rounding into the
#  measurement error, and those are different things with different causes.
# =============================================================================
class Resonator(Bus):
    def __init__(self):
        Bus.__init__(self, 0x40400000)
        self.f0_q = None    # centre frequency implied by the rounded coefficients
        self.q_q  = None    # Q implied by the rounded coefficients
    def config(self, f0, Q, delay=15, oshift=7):
        w0 = 2.0 * np.pi * f0 / FS_CLK
        r  = 1.0 - np.pi * (f0 / FS_CLK) / Q     # pole radius <-> Q, see module header
        ci = int(round(r * np.cos(w0) * 131072))  # x 2^17: coefficients are Q1.17
        cs = int(round(r * np.sin(w0) * 131072))
        self.w(0x04, ci & 0x3FFFF)
        self.w(0x08, cs & 0x3FFFF)
        self.w(0x0C, delay)          # emulate the real loop's ~15 clocks (119.87 ns).
                                     # NEVER 0: 0 silently means 256 (module header)
        self.w(0x10, oshift)         # output = state >> oshift; ~log2(Q) is right
        # what the ROUNDED coefficients actually build:
        rq        = np.hypot(ci, cs) / 131072.0
        self.f0_q = np.arctan2(cs, ci) / (2.0 * np.pi) * FS_CLK
        self.q_q  = np.pi * (self.f0_q / FS_CLK) / (1.0 - rq)
    def enable(self):  self.w(0x00, 1)
    def disable(self): self.w(0x00, 0)
    def clear(self):
        """Zero the state and the sticky clip flag, stay enabled."""
        self.w(0x00, 3)
    def clipped(self):
        return bool(self.r(0x14) & 1)
    def peek(self):
        return s32(self.r(0x18)), s32(self.r(0x1C))


# =============================================================================
#  PHASE SHIFTER (0x40300000) - the controller. See red_pitaya_phaseshift.v.
#
#  Output = A cos(phi) x I + A sin(phi) x Q, where Q is the input delayed a
#  quarter period. The host does all the trigonometry and writes two Q2.14
#  coefficients. Two module-specific corrections, both from its header:
#
#  1. The quarter-period delay D = fclk/(4 f) is only 0.78 clocks at 40 MHz -
#     unbuildable, and a delay of 0 is not allowed. Adding whole signal
#     periods changes nothing physically, so use D = fclk/(4f) + k fclk/f and
#     pick the k whose fractional part is closest to 0 or 1, where the two-tap
#     interpolator is most accurate. At 40 MHz: k = 2, D = 7.031.
#
#  2. The datapath has a fixed 2-sample latency, a pure transport delay of
#     360 x 2 x f/fclk degrees (230.4 at 40 MHz). Subtracting it from the
#     requested phase makes the delivered phase the requested one.
#
#  OPEN LOOP in this notebook means coefficients (0, 0) - output exactly zero.
#  NOT the bypass bit: bypass passes the input THROUGH, which here would be
#  unity-gain feedback, not no feedback.
# =============================================================================
class PhaseShift(Bus):
    LATENCY = 2
    def __init__(self):
        Bus.__init__(self, 0x40300000)
    def config_delay(self, f):
        best = None
        for k in range(0, 9):
            D = FS_CLK / (4.0 * f) + k * FS_CLK / f
            if D < 1.0 or D > 1000:
                continue
            frac = D - np.floor(D)
            score = min(frac, 1.0 - frac)     # interpolator error peaks at 0.5
            if best is None or score < best[0]:
                best = (score, int(np.floor(D)), frac)
        self.w(0x08, best[1])                        # whole-cycle delay
        self.w(0x0C, int(round(best[2] * 16384)))    # fractional part, Q0.14
        self.w(0x10, 0)                              # bypass OFF - see note above
        return best[1], best[2]
    def set(self, A, phi_deg, f=F0):
        phi = phi_deg - 360.0 * self.LATENCY * f / FS_CLK    # latency correction
        ci  = int(round(A * np.cos(np.radians(phi)) * 16384))
        cq  = int(round(A * np.sin(np.radians(phi)) * 16384))
        self.w(0x00, ci & 0xFFFF)
        self.w(0x04, cq & 0xFFFF)
    def off(self):
        self.set(0.0, 0.0)


# =============================================================================
#  SIGNAL GENERATOR (0x40200000, stock Red Pitaya module) - the stimulus.
#
#  It replays a 16384-entry table at a programmable rate. Load ONE sine cycle
#  once; the frequency is then a single register write: the read pointer has
#  16 fractional bits, so a full cycle spans 16384 x 2^16 = 2^30 pointer
#  units and  f = step x fclk / 2^30  - about 0.12 Hz resolution.
#
#  Register recipe (from red_pitaya_asg.v): 0x04 amplitude (0x2000 = x1.0),
#  0x08 table size, 0x10 step, buffer at 0x10000. Control 0x00: bit 6 resets
#  the pointer, 0x11 = free-run with wraparound. Writes go one at a time, as
#  everywhere on this bus.
# =============================================================================
class Generator(Bus):
    def __init__(self):
        Bus.__init__(self, 0x40200000, span=0x20000)
    def load_sine(self):
        for i in range(16384):
            v = int(round(8191.0 * np.sin(2.0 * np.pi * i / 16384.0)))
            self.w(0x10000 + 4 * i, v & 0x3FFF)
        self.w(0x08, (16384 << 16) - 1)    # table size, fixed-point, inclusive
        self.w(0x0C, 0)                    # start replaying from entry 0
    def amp(self, frac):
        """Output amplitude as a fraction of full scale, 0.0 .. 1.0."""
        self.w(0x04, int(round(frac * 8192)) & 0x3FFF)
    def freq(self, f):
        self.w(0x10, int(round(f / FS_CLK * (1 << 30))))
    def run(self):
        self.w(0x00, 0x40)     # reset the read pointer
        self.w(0x00, 0x11)     # free-run, wrap at the end of the table
    def stop(self):
        self.amp(0.0)
        self.w(0x00, 0xC0)     # hold in reset with the output forced to zero


acq, reso, ps, gen = Acq(), Resonator(), PhaseShift(), Generator()
print('drivers ready: acquisition, resonator, phase shifter, generator')

In [ ]:
# =============================================================================
#  STEP 0 - Is the resonator actually in the chip, and does it sit quietly?
#
#  An older bitstream has a STUB at this address: it answers every read with
#  zero and looks politely dead. Running the sweep against a
#  stub would produce flat nonsense with no error anywhere - the "loading an
#  old bitstream looks exactly like success" trap, which is trap number one
#  in START_HERE. So: write a recognisable value, read it back, and refuse
#  to continue if it does not return.
# =============================================================================
probe = 0x15555                       # alternating bits, sign-positive in 18
reso.w(0x04, probe)
back = reso.r(0x04) & 0x3FFFF
if back != probe:
    print('*** THE RESONATOR IS NOT IN THIS BITSTREAM ***')
    print('    wrote 0x%05X to its coefficient register, read back 0x%05X' % (probe, back))
    print('    This bitstream predates the resonator. Rebuild and reload:')
    print('      1. PC, Vivado Tcl console:  source C:/rp/prj/v0.94/Test_Project/build_project.tcl')
    print('      2. copy the bitstream across (BITSTREAM_WORKFLOW.pdf), load it LAST')
    print('      3. board:  sudo python3 check_system.py   - fingerprint must match')
    raise SystemExit('stopping: resonator absent')

probe_ps = ps.r(0x00)                 # phase shifter presence: it always answers
print('resonator present at 0x40400000, phase shifter answers at 0x40300000')
print()

# --- configure the resonator as the 40 MHz coil --------------------------
reso.config(F0, Q_SET)                # delay=15 clocks emulates the 119.87 ns loop
print('asked for      : f0 = %.0f Hz   Q = %.0f' % (F0, Q_SET))
print('chip will build: f0 = %.1f Hz   Q = %.2f   (implied by the rounded' % (reso.f0_q, reso.q_q))
print('                 18-bit coefficients - the KNOWN ANSWER every later')
print('                 step is judged against)')
print()

# --- quadrature delay for the phase shifter at this frequency ------------
dly, frac = ps.config_delay(F0)
print('phase shifter quarter-period delay: %d clocks + %.4f fractional' % (dly, frac))
if dly == 0:
    raise SystemExit('stopping: delay clamp would engage - wrong frequency plan')

# --- everything quiet, then a background reading -------------------------
ps.off()                              # coefficients (0,0): NO feedback
gen.load_sine()                       # ~16k single writes; a few seconds, once
gen.amp(0.0)
gen.run()                             # running but silent
reso.enable()
reso.clear()

acq.start(dec_log2=10)                # R = 1024: 122 kHz output, filter active
acq.tune(F0)
time.sleep(0.2)
z_bg = acq.measure(64)                # the chain's residual DC offset. Every
                                      # sweep point subtracts this complex
                                      # number, so it cannot masquerade as
                                      # signal amplitude anywhere later
u, v = reso.peek()
print()
print('silence check: resonator state (u, v) = (%d, %d)  [expect ~0]' % (u, v))
print('background    : |z_bg| = %.3e       [the voltmeter zero]' % abs(z_bg))
print('clip flag     : %s                    [must be False]' % reso.clipped())
print()
ok0 = (abs(u) < 16) and (abs(v) < 16) and not reso.clipped()
print('STEP 0:', 'PASS' if ok0 else 'FAIL - the resonator is not sitting quietly')

---
## Step 1 — Open-loop sweep: does the resonator have the Q it was told to have?

This is a network analyser built from parts already proven: the generator supplies a tone,
the resonator responds, and the acquisition system — tuned to *exactly* the drive
frequency, so the reading is a steady DC number — records how big the response is. Step
the tone across the resonance and the amplitudes trace the resonance curve.

The fit is a Lorentzian, done with a trick that needs no fitting library: the *reciprocal*
of a Lorentzian power curve is a parabola in frequency, so an ordinary weighted quadratic
fit recovers centre, width and height. Points below 5% of the peak are excluded (their
reciprocals are all noise), and the rest are weighted by their power so the well-measured
points near the peak dominate.

**Passes if** the fitted centre and Q match the values implied by the rounded
coefficients — the known answer from Step 0 — to 0.2 linewidths and 5% respectively, with
the clip flag clear. The module's arithmetic promises better than 1% on Q; the 5% gate
leaves room for the measurement's own noise without ever letting a wrong resonator pass.

> **Real coil.** `F_GRID` is built from `reso.f0_q` and the known linewidth, so it already
> knows where to look. On the real coil this becomes a two-part job: a wide coarse sweep to
> find the resonance at all, then this fine sweep around it. The fit itself is unchanged —
> it does not care where the peak came from.


In [ ]:
import matplotlib.pyplot as plt

DRIVE = 0.15          # generator amplitude, fraction of full scale. Modest on
                      # purpose: narrowing feedback will RAISE the resonator's
                      # amplitude later, and headroom must be left for that.

LW    = reso.f0_q / reso.q_q              # expected linewidth, ~222 kHz
F_GRID  = reso.f0_q + np.linspace(-2.5 * LW, 2.5 * LW, 41)
F_QUICK = reso.f0_q + np.linspace(-2.5 * LW, 2.5 * LW, 31)


def measure_amp(f):
    """Amplitude of the resonator's response at drive frequency f."""
    gen.freq(f)
    acq.tune(f)              # signal lands at DC - see the driver cell for why
    time.sleep(SETTLE)
    return abs(acq.measure() - z_bg)


def sweep(freqs):
    """One amplitude-vs-frequency sweep, with the clip check the module's
    header demands: if the state saturated ANYWHERE the curve is fiction."""
    reso.clear()
    a = np.array([measure_amp(f) for f in freqs])
    if reso.clipped():
        raise RuntimeError('resonator CLIPPED during the sweep - reduce DRIVE '
                           'or feedback gain; this curve is not physics')
    return a


def _lorentz_core(f, p, guard=0.05, minpts=3):
    """Centre, width, peak and BASELINE of a Lorentzian power curve.

    1/p is a parabola in f only for a pure Lorentzian. Any additive floor -
    detector noise, an amplifier pedestal - is absorbed into that parabola and
    inflates the fitted width. That is negligible where the peak towers over
    the floor and severe where it does not, because peak power falls as the
    square of the Q ratio: a floor worth 2% of the open-loop peak is worth 32%
    of a 4x-broadened one. Uncorrected bias, measured: +18% in linewidth at
    Q_eff/Q0 = 1, +26% at 0.5, +153% at 0.25.

    The floor is found by search, not from the wings. For each trial floor:
    subtract it, fit, rebuild the full Lorentzian-plus-floor, and keep the
    trial that reproduces the measured points best. A wing estimate needs
    points beyond ~1.5 linewidths, which a broad line does not have on this
    grid - exactly the case that needs correcting most. The search uses every
    point, so it works at any width. Two passes: coarse over [0, min(p)], then
    fine around the winner.

    Returns (f0, linewidth, peak_above_baseline, baseline).
    """
    f = np.asarray(f, dtype=float)
    p = np.asarray(p, dtype=float)
    pmax = float(np.max(p))
    u = f - f.mean()

    def _try(b):
        q = p - b
        keep = q > guard * q.max()
        if int(np.count_nonzero(keep)) < minpts:
            return None
        A, B, C = np.polyfit(u[keep], 1.0 / q[keep], 2, w=q[keep])
        if A <= 0.0:
            return None                    # parabola opens the wrong way
        u0 = -B / (2.0 * A)
        lw = 2.0 * np.sqrt(max(C / A - u0 ** 2, 1e-30))
        p0 = 1.0 / max(C - A * u0 ** 2, 1e-300)
        f0 = f.mean() + u0
        mod = p0 / (1.0 + (2.0 * (f - f0) / lw) ** 2) + b
        return float(np.sqrt(np.mean(((p - mod) / pmax) ** 2))), (f0, lw, p0, b)

    hi = 0.999 * float(np.min(p))
    best, best_b = _try(0.0), 0.0           # b = 0 is the uncorrected fit
    if hi > 0.0:
        for b in np.linspace(0.0, hi, 17):
            got = _try(b)
            if got is not None and (best is None or got[0] < best[0]):
                best, best_b = got, b
        w = hi / 16.0
        for b in np.linspace(max(best_b - w, 0.0), min(best_b + w, hi), 17):
            got = _try(b)
            if got is not None and (best is None or got[0] < best[0]):
                best, best_b = got, b
    if best is None:
        raise ValueError('fit window too narrow: %d points'
                         % int(np.count_nonzero(p > guard * p.max())))
    f0, lw, p0, base = best[1]
    return f0, lw, p0, base


def fit_lorentzian(f, a):
    """Returns (centre, linewidth, Q, peak_amplitude).

    Power p = a^2 follows p = P0 / (1 + (2(f-f0)/LW)^2), so 1/p is a parabola
    in f - once any additive floor has been taken out of p first, which is
    what _lorentz_core does. On a driven sweep like this one the floor is far
    below the peak and the correction is inert; it earns its keep on the noise
    sweeps in Stages 9 and 10."""
    f0, lw, p0, _ = _lorentz_core(f, a ** 2, guard=0.05, minpts=3)
    return f0, lw, f0 / lw, np.sqrt(p0)


# --- the open-loop measurement ----------------------------------------------
ps.off()                                   # feedback OFF: coefficients (0,0)
gen.amp(DRIVE)
amps_ol = sweep(F_GRID)
f0_ol, lw_ol, q_ol, pk_ol = fit_lorentzian(F_GRID, amps_ol)

plt.figure(figsize=(9, 4.5))
ff = np.linspace(F_GRID[0], F_GRID[-1], 400)
fit = pk_ol / np.sqrt(1.0 + (2.0 * (ff - f0_ol) / lw_ol) ** 2)
plt.plot((F_GRID - reso.f0_q) / 1e3, amps_ol, 'o', ms=4, label='measured')
plt.plot((ff - reso.f0_q) / 1e3, fit, '-', lw=1, label='Lorentzian fit')
plt.xlabel('frequency - f0 [kHz]'); plt.ylabel('response amplitude')
plt.title('Open-loop resonance:  Q asked %.2f,  Q measured %.2f' % (reso.q_q, q_ol))
plt.grid(alpha=0.4); plt.legend(); plt.tight_layout(); plt.show()

print('centre   : asked %.1f Hz   measured %.1f Hz   (off by %.0f Hz = %.3f linewidths)'
      % (reso.f0_q, f0_ol, f0_ol - reso.f0_q, abs(f0_ol - reso.f0_q) / lw_ol))
print('linewidth: %.0f Hz' % lw_ol)
print('Q        : asked %.2f   measured %.2f   (%+.2f%%)'
      % (reso.q_q, q_ol, 100.0 * (q_ol / reso.q_q - 1.0)))
print()
ok1 = (abs(f0_ol - reso.f0_q) < 0.2 * lw_ol) and (abs(q_ol / reso.q_q - 1.0) < 0.05)
print('STEP 1:', 'PASS' if ok1 else 'FAIL - the plant is not the resonator we asked for')

---
## Step 2 — The phase sweep: feedback changes the damping, and the phase chooses how

Now close the loop: the phase shifter takes the resonator's output, rotates it by phi,
scales it by a small gain, and adds it back into the drive. What happens depends on where
the returned copy lands relative to the resonator's own motion:

- **In phase with the motion's growth** — the feedback replaces energy the resonator is
  losing. Damping falls, the ring lasts longer, the line gets **narrower**.
- **Opposed** — the feedback drains energy. Damping rises, the line gets **broader**.
- **A quarter turn off** — the feedback neither adds nor removes energy but pushes the
  oscillation faster or slower: the **centre frequency shifts**, the width unchanged.

Every intermediate phase is a mixture, which is why sweeping phi traces two sinusoids:
linewidth follows a cosine, centre follows a sine, a quarter turn apart, and — because
they are two projections of one loop gain — the centre modulation must be **half** the
linewidth modulation. That factor of one half is not a fitting choice; the physics fixes
it, so the fit reproducing it is evidence the loop is doing what the theory says (this is
the sweep in Herkenhoff *et al.*, there performed on the real trap resonator).

The phase where the line is narrowest also cannot be predicted trivially — the loop's
transport delay (the emulated 119.87 ns is 286 degrees at 40 MHz, plus the phase
shifter's own latency) rotates the whole pattern. The fit finds it; that measured angle
is precisely the number the real experiment needs from this sweep.

**Passes if** both sinusoids fit with small residuals and the modulation ratio is 0.5
within measurement error.

> **Real coil.** Unchanged in principle: the phase that narrows the line is a property of
> the loop, not of the plant, and is found by sweeping. Expect it to sit somewhere else,
> because the cable and amplifier add their own delay on top of the 119.87 ns measured here.


In [ ]:
A_FB  = 0.15                       # feedback gain for the phase sweep - small on
                                   # purpose: the point is a clean sinusoid, not
                                   # the deepest possible narrowing
PHIS  = np.arange(0.0, 360.0, 15.0)          # 24 phases round the full circle

lws, fcs = [], []
for phi in PHIS:
    ps.set(A_FB, phi)
    a = sweep(F_QUICK)                       # clip-checked inside
    f0_i, lw_i, _, _ = fit_lorentzian(F_QUICK, a)
    lws.append(lw_i); fcs.append(f0_i)
ps.off()                                     # loop open again
lws = np.array(lws); fcs = np.array(fcs)


def fit_sinusoid(phi_deg, y):
    """Least-squares fit of  y = m0 + amp*cos(phi - phi0).

    Written as m0 + mc*cos(phi) + ms*sin(phi) it is LINEAR in (m0, mc, ms),
    so the normal equations solve it exactly - no iteration, no library
    beyond numpy, nothing to fail to converge."""
    c = np.cos(np.radians(phi_deg)); s = np.sin(np.radians(phi_deg))
    M = np.column_stack([np.ones_like(c), c, s])
    m0, mc, ms = np.linalg.solve(M.T.dot(M), M.T.dot(y))
    amp  = np.hypot(mc, ms)
    phi0 = np.degrees(np.arctan2(ms, mc))
    resid = y - (m0 + mc * c + ms * s)
    return m0, amp, phi0, np.sqrt(np.mean(resid ** 2))


lw0, lw_amp, lw_phi0, lw_rms = fit_sinusoid(PHIS, lws)
fc0, fc_amp, fc_phi0, fc_rms = fit_sinusoid(PHIS, fcs)

K          = lw_amp / lw0                  # the loop gain, straight from the data
PHI_NARROW = (lw_phi0 + 180.0) % 360.0     # linewidth = lw0*(1 - K cos(...)) is
                                           # SMALLEST where the cosine is +1
PHI_BROAD  = lw_phi0 % 360.0
ratio      = fc_amp / lw_amp               # physics says exactly 0.5

fig, ax = plt.subplots(2, 1, figsize=(9, 6.5), sharex=True)
pp = np.linspace(0, 360, 400)
ax[0].plot(PHIS, lws / 1e3, 'o', ms=4)
ax[0].plot(pp, (lw0 + lw_amp * np.cos(np.radians(pp - lw_phi0))) / 1e3, '-', lw=1)
ax[0].axvline(PHI_NARROW, color='g', ls=':', label='narrowest, phi = %.0f' % PHI_NARROW)
ax[0].set_ylabel('linewidth [kHz]'); ax[0].grid(alpha=0.4); ax[0].legend()
ax[0].set_title('Feedback phase sweep at gain %.2f  (the Herkenhoff sweep, emulated)' % A_FB)
ax[1].plot(PHIS, (fcs - reso.f0_q) / 1e3, 'o', ms=4)
ax[1].plot(pp, (fc0 - reso.f0_q + fc_amp * np.cos(np.radians(pp - fc_phi0))) / 1e3, '-', lw=1)
ax[1].set_xlabel('feedback phase [deg]'); ax[1].set_ylabel('centre - f0 [kHz]')
ax[1].grid(alpha=0.4)
plt.tight_layout(); plt.show()

print('loop gain K              : %.3f      (from the linewidth modulation)' % K)
print('narrowing phase          : %.1f deg  (transport delay folded in - this is' % PHI_NARROW)
print('                                      the number the real experiment needs)')
print('linewidth sinusoid rms   : %.1f Hz on a %.0f Hz modulation' % (lw_rms, lw_amp))
print('centre    sinusoid rms   : %.1f Hz on a %.0f Hz modulation' % (fc_rms, fc_amp))
print('centre/linewidth ratio   : %.3f      (physics fixes this at 0.500)' % ratio)
print()
ok2 = (lw_rms < 0.15 * lw_amp) and (fc_rms < 0.25 * fc_amp) and (0.35 < ratio < 0.65)
print('STEP 2:', 'PASS' if ok2 else 'FAIL - the loop is not behaving like the theory')

---
## Step 3 — The payoff: narrow the line, broaden the line, on demand

Step 2 found the two special phases. Now sit on each and measure the full resonance
curve, overlaid on the open-loop one. This is Q-control — the reason the feedback system
exists: at the narrowing phase the detection resonance rings longer and the effective Q
rises; at the broadening phase it falls.

The check is quantitative, using the loop gain K that Step 2 measured — not a fitted
afterthought: `Q_eff = Q0 / (1 -+ K)`. Every number in that prediction was measured in an
*earlier* step, so agreement here means three independent measurements are consistent.

**Passes if** both effective Qs match the prediction within 20%, with no clipping —
watch the narrowed curve: its peak amplitude *should* rise by about `1/(1-K)`, and the
clip check inside `sweep()` guards the headroom.

> **Real coil.** Unchanged. This step compares the narrowing and broadening phases against
> each other, which needs no known answer — one of the checks worth keeping.


In [ ]:
results = {}
for name, phi in (('narrow', PHI_NARROW), ('broad', PHI_BROAD)):
    ps.set(A_FB, phi)
    a = sweep(F_GRID)
    results[name] = (a,) + fit_lorentzian(F_GRID, a)
ps.off()

plt.figure(figsize=(9, 5))
plt.plot((F_GRID - reso.f0_q) / 1e3, amps_ol, 'o-', ms=3, lw=0.7,
         label='open loop        Q = %.1f' % q_ol)
plt.plot((F_GRID - reso.f0_q) / 1e3, results['narrow'][0], 's-', ms=3, lw=0.7,
         label='narrowing phase  Q = %.1f' % results['narrow'][3])
plt.plot((F_GRID - reso.f0_q) / 1e3, results['broad'][0], 'v-', ms=3, lw=0.7,
         label='broadening phase Q = %.1f' % results['broad'][3])
plt.xlabel('frequency - f0 [kHz]'); plt.ylabel('response amplitude')
plt.title('Q-control: one resonator, three linewidths, chosen by feedback phase')
plt.grid(alpha=0.4); plt.legend(); plt.tight_layout(); plt.show()

q_n_pred = q_ol / (1.0 - K)
q_b_pred = q_ol / (1.0 + K)
q_n, q_b = results['narrow'][3], results['broad'][3]

print('               predicted   measured')
print('  narrowed Q :   %6.1f     %6.1f   (%+.1f%%)' % (q_n_pred, q_n, 100*(q_n/q_n_pred-1)))
print('  broadened Q:   %6.1f     %6.1f   (%+.1f%%)' % (q_b_pred, q_b, 100*(q_b/q_b_pred-1)))
print()
print('  (predictions use only numbers measured in Steps 1 and 2 - nothing here')
print('   was fitted to make this work)')
print()
ok3 = (abs(q_n / q_n_pred - 1.0) < 0.20) and (abs(q_b / q_b_pred - 1.0) < 0.20)
print('STEP 3:', 'PASS' if ok3 else 'FAIL - Q does not respond to feedback as predicted')

---
## Step 4 — The gain sweep, and where the cliff edge is

At the narrowing phase, more gain means more narrowing: `linewidth/linewidth0 = 1 - K`,
with K proportional to the gain setting. That is a straight line pointing at zero — and
zero linewidth means the feedback fully replaces the resonator's losses, at which point
it stops being an amplifier of ring-time and becomes an oscillator. The real experiment
operates deliberately close to that edge (it is where the narrowest lines are), so
knowing exactly where the edge sits, and proving the approach to it is linear, is the
last thing this stage has to establish.

The sweep stays at or below gain 0.30 — comfortably inside the line's predicted safe
region — and *extrapolates* to the threshold rather than walking onto it. If the line is
straight, the extrapolation is trustworthy; that is what the residual check verifies.

**Passes if** the relationship is linear (small residuals), narrowing increases with
gain, and the intercept at zero gain is 1 — no narrowing without feedback, which doubles
as a check that nothing is leaking around the phase shifter.

> **Real coil.** Unchanged, and this is the most valuable check that survives: K must grow
> in proportion to feedback gain. It tests the loop against itself rather than against a
> number you supplied.


In [ ]:
GAINS = np.array([0.05, 0.10, 0.15, 0.20, 0.25, 0.30])

rel = []
for A in GAINS:
    ps.set(A, PHI_NARROW)
    a = sweep(F_QUICK)
    _, lw_i, _, _ = fit_lorentzian(F_QUICK, a)
    rel.append(lw_i / lw_ol)              # linewidth relative to open loop
ps.off()
rel = np.array(rel)

slope, intercept = np.polyfit(GAINS, rel, 1)
resid_rms = np.sqrt(np.mean((rel - (intercept + slope * GAINS)) ** 2))
a_crit    = -intercept / slope if slope < 0 else float('inf')

plt.figure(figsize=(8, 4.5))
gg = np.linspace(0, min(a_crit * 1.05, GAINS[-1] * 2), 200)
plt.plot(GAINS, rel, 'o', ms=5, label='measured')
plt.plot(gg, intercept + slope * gg, '-', lw=1, label='linear fit')
plt.axhline(0, color='k', lw=0.5)
if np.isfinite(a_crit):
    plt.axvline(a_crit, color='r', ls=':',
                label='oscillation threshold ~ %.2f' % a_crit)
plt.xlabel('feedback gain setting A'); plt.ylabel('linewidth / open-loop linewidth')
plt.title('Approach to the self-oscillation threshold, measured safely from below')
plt.grid(alpha=0.4); plt.legend(); plt.tight_layout(); plt.show()

print('linearity        : rms residual %.4f on a span of %.3f' % (resid_rms, rel[0] - rel[-1]))
print('zero-gain value  : %.3f            (physics says 1.000)' % intercept)
print('threshold (extrapolated): gain ~ %.3f - STAY BELOW ~%.2f in any later use'
      % (a_crit, 0.7 * a_crit))
print()
ok4 = (slope < 0) and (resid_rms < 0.03) and (0.9 < intercept < 1.1)
print('STEP 4:', 'PASS' if ok4 else 'FAIL - narrowing is not linear in gain')

---
## Stage 8 summary

| Step | What it proves |
|---|---|
| 0 | The resonator is in the chip, configured, and silent when undriven |
| 1 | The plant is exactly the resonator asked for: Q measured against the coefficient-implied known answer |
| 2 | Closing the loop modulates linewidth and centre as two sinusoids with the physics-fixed 2:1 ratio; the narrowing phase is *measured*, transport delay and all |
| 3 | Q-control works: effective Q follows `Q0/(1 -+ K)` using only previously measured numbers |
| 4 | Narrowing is linear in gain, the zero-gain intercept is 1, and the self-oscillation threshold is located from a safe distance |

**With this passing, the feedback system is verified against a known plant.** The
instrument (Stages 1–7) and now the controller (Stage 8) have both been proven with
known answers. What remains between here and physics is only the real world:

- **Swap the plant.** Disable the resonator (one register write) and the same
  measurement, with the same notebooks, runs against the real coil through the actual
  cables — where f0 and Q are no longer known, but the *method* now is.
- The narrowing phase found here is emulation-specific (it contains the emulated delay);
  on the real coil, re-run Step 2 to find the real one. That sweep is the calibration
  the experiment needs anyway.
- **Trigger synchronisation** (starting captures in step with the feedback sequencer)
  remains unbuilt, as noted in `HANDOVER.md`.

A caution worth repeating from the module's header: the resonator **amplifies by about
Q at resonance**. Q = 180 leaves ample headroom, but anyone raising Q towards 10,000 for
a cryogenic emulation must reduce the drive and re-check the clip flag — the table in
`red_pitaya_resonator.v` says exactly how far each Q can be driven.

> **Real coil.** Harmless either way. With the resonator never enabled, `reso.disable()`
> does nothing — but leaving the generator stopped and the feedback open is still the right
> state to finish in.


In [ ]:
# Leave the board exactly as a power-cycle would: feedback open, generator
# silent and held in reset, resonator disabled (the input multiplexer then
# passes the real ADC through, bit-for-bit), acquisition off.
ps.off()
gen.stop()
reso.disable()
acq.w(0x00, 0)
print('all quiet: feedback off, generator stopped, resonator disabled,')
print('the board is hearing its real inputs again')